# Kiểm định & Đối chiếu Dóng hàng Hán – Việt bằng Bertalign (Đề tài CV09)

Notebook này sử dụng nguyên bản **[bfsujason/bertalign](https://github.com/bfsujason/bertalign)** (mô hình nhúng đa ngôn ngữ **LaBSE** + FAISS + DP 2 bước) trên **Google Colab GPU** để kiểm chứng độc lập toàn bộ **4 quyển / 6.886 cặp câu** của đề tài `CV09`:
1. **Sắc tu Bách Trượng thanh quy – Quyển 4** (`q4_parallel.tsv` – 482 cặp)
2. **Sắc tu Bách Trượng thanh quy – Quyển 7** (`q7_parallel.tsv` – 404 cặp)
3. **Triệu Châu Ngữ Lục – Quyển 14** (`TrieuChauNguLuc_parallel.tsv` – 1.039 cặp)
4. **Giải Thoát Đạo Luận – 12 Quyển** (`GiaiThoatDaoLuan_parallel.tsv` – 4.961 cặp)

> **Lưu ý trước khi chạy:** Trên Google Colab, vào menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU** để chạy LaBSE/Bertalign nhanh gấp ~25 lần (toàn bộ 4 quyển chỉ mất ~2–3 phút).

## Bước 1: Kiểm tra GPU, cài đặt `bertalign` và vá lỗi tương thích Python 3.11/3.12 (`googletrans`)

> Repo gốc `bertalign` import `googletrans` (thường gây xung đột `httpx`/`cgi` trên Python 3.11+). Vì chúng ta truyền tường minh `src_lang="zh", tgt_lang="vi"`, ô lệnh dưới sẽ tự động vá `bertalign/utils.py` để chạy mượt 100% trên mọi phiên bản Colab.

In [12]:
!nvidia-smi
!rm -rf /content/bertalign
!git clone -q https://github.com/bfsujason/bertalign.git /content/bertalign
!pip install -q sentence-transformers faiss-cpu numba pandas openpyxl sentence-splitter

# Vá lỗi import googletrans trong bertalign/utils.py (không cần auto-detect vì truyền rõ zh/vi)
import pathlib, re
utils_path = pathlib.Path("/content/bertalign/bertalign/utils.py")
src_code = utils_path.read_text(encoding="utf-8")
if "from googletrans import Translator" in src_code:
    patch = "try:\n    from googletrans import Translator\nexcept Exception:\n    Translator = None\n"
    src_code = src_code.replace("from googletrans import Translator", patch)
    utils_path.write_text(src_code, encoding="utf-8")
    print("[OK] Đã vá googletrans trong bertalign/utils.py")

import sys
if "/content/bertalign" not in sys.path:
    sys.path.insert(0, "/content/bertalign")

import torch
from bertalign import Bertalign
from bertalign.encoder import Encoder
print(f"[OK] Bertalign sẵn sàng! CUDA available: {torch.cuda.is_available()}")

Fri Nov 27 05:35:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             31W /   70W |    4369MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Bước 2: Nạp dữ liệu `.tsv` của 4 quyển

Chọn **1 trong 2 cách** bên dưới:
- **Cách A (Khuyên dùng – nếu đã push mới nhất lên GitHub):** Đặt `LOAD_MODE = "github"` để tự động tải thẳng từ repo `Lmt2703/NLP_Project`.
- **Cách B (Tải trực tiếp từ máy tính lên Colab):** Đặt `LOAD_MODE = "upload"` và chọn 4 tệp trong `CV09_SacTuBachTruongThanhQuy/04_dong_hang/`:
  - `q4_parallel.tsv`
  - `q7_parallel.tsv`
  - `TrieuChauNguLuc_parallel.tsv`
  - `GiaiThoatDaoLuan_parallel.tsv`

In [ ]:
import os

LOAD_MODE = "upload"  # Đổi thành "github" nếu đã push các file mới nhất lên GitHub
DATA_DIR = "/content/cv09_data"
os.makedirs(DATA_DIR, exist_ok=True)

BOOK_FILES = [
    ("BT_Q4", "q4_parallel.tsv"),
    ("BT_Q7", "q7_parallel.tsv"),
    ("TC_Q14", "TrieuChauNguLuc_parallel.tsv"),
    ("GTDL_12Q", "GiaiThoatDaoLuan_parallel.tsv"),
]

if LOAD_MODE == "github":
    !rm -rf /content/NLP_Project
    !git clone -q https://github.com/Lmt2703/NLP_Project.git /content/NLP_Project
    src_dir = "/content/NLP_Project/CV09_SacTuBachTruongThanhQuy/04_dong_hang"
    for _, fname in BOOK_FILES:
        src_f = os.path.join(src_dir, fname)
        if os.path.isfile(src_f):
            !cp "{src_f}" "{DATA_DIR}/{fname}"
else:
    from google.colab import files
    print("Hãy chọn các file *_parallel.tsv cần kiểm định (có thể chọn 1 hoặc cả 4 file):")
    uploaded = files.upload()
    for fname, content in uploaded.items():
        with open(os.path.join(DATA_DIR, fname), "wb") as f:
            f.write(content)

available_books = []
for code, fname in BOOK_FILES:
    fpath = os.path.join(DATA_DIR, fname)
    if os.path.isfile(fpath):
        n_lines = sum(1 for _ in open(fpath, encoding="utf-8"))
        available_books.append((code, fname, fpath))
        print(f"  [✓] {code:<10} | {fname:<32} | {n_lines:,} cặp câu")
    else:
        print(f"  [ ] {code:<10} | {fname:<32} | (chưa có)")

Hãy chọn các file *_parallel.tsv cần kiểm định (có thể chọn 1 hoặc cả 4 file):


## Bước 3: Kiểm định ngữ nghĩa & phát hiện trượt dòng bằng bộ mã hóa LaBSE của Bertalign

Sử dụng trực tiếp `Encoder("LaBSE")` của Bertalign để:
1. Tính độ tương đồng ngữ nghĩa `sim(H_i, V_i)` cho từng cặp câu trong mỗi quyển.
2. So sánh chéo với câu liền trên (`V_{i-1}`) và câu liền dưới (`V_{i+1}`) để phát hiện chính xác các dòng bị **trượt ranh giới** (`best_neighbor - sim_current >= 0.05`).
3. Xuất file tổng hợp **`Bertalign_LaBSE_Audit.xlsx`**.

In [ ]:
import numpy as np
import pandas as pd

print(">>> Đang nạp mô hình LaBSE (backbone của Bertalign)...")
encoder = Encoder("LaBSE")

def load_tsv(path):
    pairs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\r\n").split("\t")
            if len(parts) == 2:
                pairs.append((parts[0].strip(), parts[1].strip()))
    return pairs

def encode_texts(texts):
    # Bertalign Encoder.transform() trả về (vecs, lens) đã chuẩn hóa L2
    vecs, _ = encoder.transform(texts, num_overlaps=1)
    vecs = vecs[0]
    norms = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12
    return vecs / norms

summary_rows = []
flagged_by_book = {}

for code, fname, fpath in available_books:
    pairs = load_tsv(fpath)
    hans = [h for h, _ in pairs]
    viets = [v for _, v in pairs]

    print(f"\n>>> Đang mã hóa LaBSE cho {code} ({len(pairs):,} cặp)...")
    h_emb = encode_texts(hans)
    v_emb = encode_texts(viets)

    # Cosine similarity từng cặp hiện tại
    sim_cur = np.sum(h_emb * v_emb, axis=1)
    # Cosine similarity với câu Việt liền trước / liền sau
    sim_prev = np.concatenate([[0.0], np.sum(h_emb[1:] * v_emb[:-1], axis=1)])
    sim_next = np.concatenate([np.sum(h_emb[:-1] * v_emb[1:], axis=1), [0.0]])

    flagged = []
    for i in range(len(pairs)):
        sc = float(sim_cur[i])
        sp = float(sim_prev[i])
        sn = float(sim_next[i])
        best_nb = max(sp, sn)
        nb_dir = "PREV (i-1)" if sp >= sn else "NEXT (i+1)"

        flags = []
        if best_nb - sc >= 0.05:
            flags.append(f"NGHI_TRUOT_{nb_dir}")
        if sc < 0.45:
            flags.append("DIEM_THAP(<0.45)")

        if flags:
            flagged.append({
                "Dong": i + 1,
                "Canh_bao": " | ".join(flags),
                "Sim_hien_tai": round(sc, 4),
                "Sim_dong_tren": round(sp, 4),
                "Sim_dong_duoi": round(sn, 4),
                "Chenh_lech_lan_can": round(best_nb - sc, 4),
                "Han": hans[i],
                "Viet": viets[i],
            })

    flagged_by_book[code] = pd.DataFrame(flagged)
    n_drift = sum(1 for r in flagged if "NGHI_TRUOT" in r["Canh_bao"])
    n_low = sum(1 for r in flagged if "DIEM_THAP" in r["Canh_bao"])
    summary_rows.append({
        "Quyen": code,
        "So_cap": len(pairs),
        "Sim_TB": round(float(np.mean(sim_cur)), 4),
        "Sim_Trung_vi": round(float(np.median(sim_cur)), 4),
        "Ty_le_Sim>=0.60 (%)": round(float(np.mean(sim_cur >= 0.60) * 100), 2),
        "Nghi_truot_dong": n_drift,
        "Diem_thap(<0.45)": n_low,
        "Tong_dong_can_xem": len(flagged),
    })

df_summary = pd.DataFrame(summary_rows)
display(df_summary)

audit_xlsx = "/content/Bertalign_LaBSE_Audit.xlsx"
with pd.ExcelWriter(audit_xlsx, engine="openpyxl") as writer:
    df_summary.to_excel(writer, sheet_name="TONG_HOP", index=False)
    for code, df_f in flagged_by_book.items():
        if len(df_f) > 0:
            df_f.sort_values("Sim_hien_tai").to_excel(writer, sheet_name=f"FLAG_{code}", index=False)
        else:
            pd.DataFrame([{"Thong_bao": "Không phát hiện dòng bất thường"}]).to_excel(writer, sheet_name=f"FLAG_{code}", index=False)

print(f"\n[+] Đã xuất báo cáo kiểm định LaBSE tại: {audit_xlsx}")

## Bước 4: Chạy dóng hàng độc lập bằng `Bertalign` & So sánh đồng thuận

Chạy trực tiếp `Bertalign(src, tgt, src_lang="zh", tgt_lang="vi", is_split=True)` trên từng quyển (lấy riêng cột Hán và cột Việt của file `.tsv`, để Bertalign tự dóng lại hoàn toàn độc lập rồi so sánh xem bao nhiêu phần trăm cặp Bertalign đồng thuận `1-1` khớp đúng cặp `[i] ↔ [i]` của chúng ta!).

- **Đồng thuận `1-1` (`[i] ↔ [i]`):** Bertalign xác nhận cặp hiện tại hoàn toàn chính xác.
- **Khác biệt (`1-0`, `0-1`, `1-2`, `2-1`, lệch chỉ số):** Xuất chi tiết ra file **`Bertalign_Realign_Diff.xlsx`** để bạn đối chiếu và sửa nếu cần.

In [ ]:
import bertalign.utils as bu
import bertalign.aligner as ba

def _fast_detect_lang(text):
    cjk = sum(1 for c in text[:300] if '\u4e00' <= c <= '\u9fff')
    return 'zh' if cjk > 10 else 'vi'

bu.detect_lang = _fast_detect_lang
ba.detect_lang = _fast_detect_lang
import bertalign.corelib as bc
bc.platform = 'cpu'
realign_summary = []
diff_sheets = {}

# Lưu ý: Giải Thoát Đạo Luận dài 4.961 dòng -> chia theo block ~500 dòng (khớp điểm neo chắc chắn)
# hoặc chạy trực tiếp từng block 400 dòng để Bertalign không bị tràn bộ nhớ và giữ tốc độ nhanh.
CHUNK_SIZE = 500

def run_bertalign_check(hans, viets, chunk_size=500):
    n = len(hans)
    exact_1_1 = 0
    diff_records = []
    total_beads = 0

    for start in range(0, n, chunk_size):
        end = min(n, start + chunk_size)
        src_text = "\n".join(hans[start:end])
        tgt_text = "\n".join(viets[start:end])

        aligner = Bertalign(
            src_text,
            tgt_text,
            max_align=5,
            top_k=3,
            win=5,
            skip=-0.1,
            margin=True,
            len_penalty=True,
            is_split=True,
        )
        aligner.align_sents()

        for bead in aligner.result:
            total_beads += 1
            src_idxs, tgt_idxs = bead[0], bead[1]
            # Kiểm tra xem Bertalign có đồng thuận hoàn toàn 1-1 đúng chỉ số gốc hay không
            if len(src_idxs) == 1 and len(tgt_idxs) == 1 and src_idxs[0] == tgt_idxs[0]:
                exact_1_1 += 1
            else:
                g_src = [start + k + 1 for k in src_idxs]
                g_tgt = [start + k + 1 for k in tgt_idxs]
                src_str = " ".join(hans[start + k] for k in src_idxs)
                tgt_str = " ".join(viets[start + k] for k in tgt_idxs)
                diff_records.append({
                    "Kieu_Bertalign": f"{len(src_idxs)}-{len(tgt_idxs)}",
                    "Dong_Han_goc": str(g_src),
                    "Dong_Viet_goc": str(g_tgt),
                    "Han_Bertalign": src_str,
                    "Viet_Bertalign": tgt_str,
                })

    return exact_1_1, total_beads, diff_records

for code, fname, fpath in available_books:
    pairs = load_tsv(fpath)
    hans = [h for h, _ in pairs]
    viets = [v for _, v in pairs]
    print(f"\n>>> Đang chạy Bertalign độc lập cho {code} ({len(pairs):,} cặp)...")
    exact_1_1, total_beads, diffs = run_bertalign_check(hans, viets, chunk_size=CHUNK_SIZE)
    agree_pct = round((exact_1_1 / len(pairs)) * 100, 2) if pairs else 0.0
    print(f"    -> Đồng thuận chính xác 1-1: {exact_1_1:,}/{len(pairs):,} ({agree_pct}%) | Số cụm khác biệt: {len(diffs)}")
    realign_summary.append({
        "Quyen": code,
        "Tong_cap_hien_tai": len(pairs),
        "Bertalign_dong_thuan_1_1": exact_1_1,
        "Ty_le_dong_thuan (%)": agree_pct,
        "So_cum_khac_biet": len(diffs),
    })
    diff_sheets[code] = pd.DataFrame(diffs)

df_realign = pd.DataFrame(realign_summary)
display(df_realign)

diff_xlsx = "/content/Bertalign_Realign_Diff.xlsx"
with pd.ExcelWriter(diff_xlsx, engine="openpyxl") as writer:
    df_realign.to_excel(writer, sheet_name="TONG_HOP", index=False)
    for code, df_d in diff_sheets.items():
        if len(df_d) > 0:
            df_d.to_excel(writer, sheet_name=f"DIFF_{code}", index=False)
        else:
            pd.DataFrame([{"Thong_bao": "Bertalign đồng thuận 100%"}]).to_excel(writer, sheet_name=f"DIFF_{code}", index=False)

print(f"\n[+] Đã xuất file so sánh khác biệt Bertalign tại: {diff_xlsx}")

## Bước 5: Tải 2 báo cáo kiểm định Excel về máy

In [ ]:
from google.colab import files
files.download("/content/Bertalign_LaBSE_Audit.xlsx")
files.download("/content/Bertalign_Realign_Diff.xlsx")